# 11.6 - Computer Use Agents

**Module 11 - AI Agents** | Netsetos GenAI Engineering

This notebook builds a computer-use agent from its parts: the perception-action loop over screenshots, the three ways to drive an interface (DOM distillation, vision + coordinates, the native Anthropic tool), the perception-cost tradeoff, a safety gate, and the API-first hierarchy that decides which approach to reach for. Every cell runs with no API key - the model and the "screen" are scripted so you can watch the shape of each mechanism.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - install dependencies

The one package this lesson could call for real is the Anthropic SDK; everything else in the notebook is pure Python that runs with no install. This cell is commented out so a fresh Colab or local env can enable it in one step.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" python-dotenv -q  # noqa


**Explanation**

A dependency-install cell, not code that does anything on its own. It is left commented so the notebook runs offline; uncomment it only on a fresh machine that needs the SDK.

**How the code works - step by step**
- **`anthropic>=0.40.0`** - the Anthropic Python SDK, needed only for the illustrative real computer-use call (the runnable cells never import it).
- **`python-dotenv`** - lets you load an API key from a local `.env` file instead of hardcoding it.
- **`# noqa`** - silences the linter on the `!pip` shell magic.

**In one line:** optional environment prep, off by default.

**What you'll see:** (no output - environment prep)

## Setup - configure the API key

The runnable cells are all scripted and need no key, but the two illustrative blocks (real browser-use and real Anthropic Computer Use) read `ANTHROPIC_API_KEY` from the environment. This cell wires that up without ever hardcoding a secret.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com


**Explanation**

A key-loading cell. It reads the key from the environment rather than embedding it, using `setdefault` so an already-exported key is never overwritten by the empty placeholder.

**How the code works - step by step**
- **`import os`** - gives access to the process environment.
- **`os.environ.setdefault("ANTHROPIC_API_KEY", "")`** - registers the key name and leaves any real value already present untouched; the empty string is just a placeholder.

**In one line:** load the key from the environment, never hardcode it.

**What you'll see:** (no output - environment prep)

## 1 - The perception-action loop

Every computer-use agent is one loop: take a screenshot (observe), decide a single action (think), execute it (act), then screenshot again - because the action changed the screen. Building it with a scripted screen and scripted policy shows the shape end to end, with a hard step cap, and no API key.

In [ ]:
# THE PERCEPTION-ACTION LOOP - the core of every computer-use agent.
# A computer-use agent does NOT call a clean API. It loops:
#   observe (screenshot) -> think (decide an action) -> act -> observe again.
# We script the "screen" and the "thinking" so the SHAPE runs with no key.

SCREEN = {"url": "about:blank", "page": "home", "goal_visible": False}

def screenshot(screen):                       # OBSERVE: what the agent can see
    return f"url={screen['url']} page={screen['page']} goal_visible={screen['goal_visible']}"

def decide(observation, step):                # THINK: a scripted policy (stands in for the model)
    if "about:blank" in observation:          # nothing loaded yet -> navigate
        return {"action": "goto", "url": "portal/invoices"}
    if "goal_visible=False" in observation:   # page loaded, target not shown -> click into it
        return {"action": "click", "target": "Invoices"}
    return {"action": "done", "result": "read 3 invoice totals"}   # target visible -> finish

def act(screen, action):                      # ACT: apply the action to the world
    if action["action"] == "goto":
        screen["url"] = action["url"]; screen["page"] = "invoices"
    elif action["action"] == "click":
        screen["goal_visible"] = True
    return screen

def run(screen, max_steps=8):                 # the loop, with a HARD step cap (cost + safety)
    for step in range(1, max_steps + 1):
        obs = screenshot(screen)
        action = decide(obs, step)
        print(f"  step {step}: observe[{obs}] -> {action['action']}")
        if action["action"] == "done":
            return {"steps": step, "result": action["result"]}
        screen = act(screen, action)
    return {"steps": max_steps, "result": "step cap reached"}

print("Perception-action loop (screenshot -> think -> act -> repeat):")
out = run(dict(SCREEN))
print(f"\nFinished in {out['steps']} steps: {out['result']}")
print("A function call would be ONE structured request; computer use is this whole loop over pixels.")
# Output:
# Perception-action loop (screenshot -> think -> act -> repeat):
#   step 1: observe[url=about:blank page=home goal_visible=False] -> goto
#   step 2: observe[url=portal/invoices page=invoices goal_visible=False] -> click
#   step 3: observe[url=portal/invoices page=invoices goal_visible=True] -> done
#
# Finished in 3 steps: read 3 invoice totals
# A function call would be ONE structured request; computer use is this whole loop over pixels.

**Explanation**

A tiny simulator of the whole computer-use loop. A dict stands in for the screen, a scripted policy stands in for the model, and `run` drives observe-think-act until the policy says done or the step cap fires. Read it as: this is the skeleton every later cell fills in with real perception and a real model.

**How the code works - step by step**
- **`SCREEN`** - a dict standing in for what is on screen (url, page, whether the goal is visible).
- **`screenshot()`** - the OBSERVE step; returns a string that stands in for a real image.
- **`decide()`** - the THINK step; a scripted policy that navigates, then clicks in, then finishes - standing in for the model reading pixels.
- **`act()`** - applies the chosen action to the screen dict, so the next iteration observes a *different* state.
- **`run()`** - the loop itself, with a hard `max_steps=8` cap that stops a runaway agent (the same safeguard as Lesson 11.1).

**In one line:** observe -> think -> act -> observe, capped by max_steps.

**What you'll see:** Three numbered steps printing the observation and chosen action (goto -> click -> done), then "Finished in 3 steps: read 3 invoice totals" and a reminder that a function call would be one request while this is a whole loop over pixels.

## 2 - DOM distillation for browser tasks

When the target is a web page, you should not feed the model raw HTML - it is thousands of tokens of tags, scripts, and styling. Distill it to a short numbered list of only the *interactive* elements so the model can act by index ("click [3]"), the trick behind browser-use. This cell distills a messy page and measures the token savings.

In [ ]:
# DOM DISTILLATION - why browser agents do not send raw HTML to the model.
# Raw HTML is huge and mostly noise. Distill it to a compact list of the
# INTERACTIVE elements, each with a number the agent can act on ("click 2").
import re

RAW_HTML = """
<html><head><style>.btn{color:blue}</style><script>track()</script></head>
<body><div class="wrap"><nav><ul><li><a href="/home">Home</a></li>
<li><a href="/invoices">Invoices</a></li></ul></nav>
<main><h1>Your account</h1><p>Welcome back. Manage billing below.</p>
<input type="text" name="search" placeholder="Search invoices">
<button id="download">Download all</button></main></div></body></html>
"""
INTERACTIVE = ("a", "button", "input")

def _attr(attrs, key):
    m = re.search(key + r'="([^"]*)"', attrs)
    return m.group(1) if m else ""

def distill(html_text):
    # Keep only interactive elements; assign each a stable index the model can name.
    elements = []
    for m in re.finditer(r"<(a|button|input)\b([^>]*)>(.*?)</\1>|<(input)\b([^>]*)/?>", html_text):
        tag = m.group(1) or m.group(4)
        attrs = m.group(2) or m.group(5) or ""
        text = (m.group(3) or "").strip()
        label = text or _attr(attrs, "placeholder") or _attr(attrs, "name") or _attr(attrs, "href")
        elements.append((tag, label))
    return elements

def render_for_model(elements):
    return "\n".join(f"[{i}] {tag}: {label}" for i, (tag, label) in enumerate(elements))

els = distill(RAW_HTML)
compact = render_for_model(els)
print("Distilled interactive elements the model actually needs:")
print(compact)
raw_tok = len(RAW_HTML) // 4          # rough token estimate: ~4 chars/token
compact_tok = len(compact) // 4
print(f"\nRaw HTML ~{raw_tok} tokens  ->  distilled ~{compact_tok} tokens"
      f"  ({100 - round(100 * compact_tok / raw_tok)}% smaller)")
print("The agent clicks by index ([3] button: Download all), never re-reading the markup.")
# Output:
# Distilled interactive elements the model actually needs:
# [0] a: Home
# [1] a: Invoices
# [2] input: Search invoices
# [3] button: Download all
#
# Raw HTML ~100 tokens  ->  distilled ~19 tokens  (81% smaller)
# The agent clicks by index ([3] button: Download all), never re-reading the markup.

**Explanation**

A page-compression routine, not a model call. It regex-scans HTML, keeps only the clickable/typeable tags, labels each, and numbers them so the model names an element instead of guessing a pixel. The point is the token drop and the stable index, both visible in the output.

**How the code works - step by step**
- **`RAW_HTML`** - a sample page with nav, prose, styles, and a script - mostly noise for an agent.
- **`_attr()`** - a helper that pulls a named attribute (placeholder / name / href) out of a tag's attribute string.
- **`distill()`** - keeps only `a` / `button` / `input` tags, drops scripts, styles, and prose, and picks a human-readable label for each.
- **`render_for_model()`** - formats the survivors as a numbered list (`[0] a: Home`), giving each a stable index the model can name.
- **Token estimate** - compares raw vs distilled length at ~4 chars/token to show the shrink.

**In one line:** strip the page to its numbered interactive elements, act by index.

**What you'll see:** A four-line distilled list ([0] Home, [1] Invoices, [2] Search invoices, [3] Download all), then "Raw HTML ~100 tokens -> distilled ~19 tokens (81% smaller)" and a note that the agent clicks by index, never re-reading the markup.

## 3 - Vision + coordinates and the scaling trap

When there is no DOM - a canvas game, a custom UI, a desktop app - you send a screenshot and the model returns an (x, y) to click. The notorious bug: the API downscales the image first, so the model's coordinates live in the *shrunk* space, and clicking them raw on your real screen misses. This cell runs Anthropic's own scale-factor formula to show the bug and the one-line fix.

In [ ]:
# THE COORDINATE-SCALING TRAP - the #1 bug in vision-based computer use.
# The API DOWNSCALES big screenshots, and the model returns coordinates in the
# space of the image it SAW. If you click those raw on your real screen, you miss.
# Fix (Anthropic's own formula): resize yourself, then scale coordinates back.
import math

LONG_EDGE = 1568          # earlier-model long-edge cap (Sonnet 5 / Opus 4.8 allow 2576)
MAX_PIXELS = 1_150_000    # ~1.15 megapixel cap

def scale_factor(w, h):
    return min(1.0, LONG_EDGE / max(w, h), math.sqrt(MAX_PIXELS / (w * h)))

native_w, native_h = 1920, 1080
s = scale_factor(native_w, native_h)
sent_w, sent_h = round(native_w * s), round(native_h * s)
print(f"Native screen {native_w}x{native_h} -> scale {s:.3f} -> send {sent_w}x{sent_h}")

# The model looks at the sent (downscaled) image and says "click the button at:"
model_click = (512, 300)                      # coordinates in the SENT image space
wrong = model_click                           # BUG: click straight away on the real screen
right = (round(model_click[0] / s), round(model_click[1] / s))  # FIX: scale back to native

print(f"\nModel returns click {model_click} (in the {sent_w}x{sent_h} image it saw)")
print(f"  WRONG - click raw on native: {wrong}   (off by ~{1 / s:.2f}x, misses the target)")
print(f"  RIGHT - scale back to native: {right}")
print("Rule: send at a known size, keep the scale factor, multiply coordinates by 1/scale.")
# Output:
# Native screen 1920x1080 -> scale 0.745 -> send 1430x804
#
# Model returns click (512, 300) (in the 1430x804 image it saw)
#   WRONG - click raw on native: (512, 300)   (off by ~1.34x, misses the target)
#   RIGHT - scale back to native: (688, 403)
# Rule: send at a known size, keep the scale factor, multiply coordinates by 1/scale.

**Explanation**

A coordinate-mapping demo, not a model call. It computes the exact downscale factor Anthropic applies, shows that the model's click is in the sent-image space, and contrasts clicking it raw (wrong) against multiplying by 1/scale (right). The lesson is arithmetic: keep the scale factor and map coordinates back to native.

**How the code works - step by step**
- **`scale_factor()`** - Anthropic's formula: cap the long edge (`LONG_EDGE`) and the total megapixels (`MAX_PIXELS`), take the smallest ratio.
- **`sent_w, sent_h`** - the 1920x1080 screen is actually sent at ~1430x804, so that is the space the model sees.
- **`model_click`** - a coordinate the model returns, expressed in that downscaled image.
- **`wrong`** - clicking that coordinate straight on the native screen; off by ~1/scale, misses the target.
- **`right`** - dividing by the scale factor to map the coordinate back to native pixels - the fix.

**In one line:** send at a known size, keep the scale factor, multiply returned coordinates by 1/scale.

**What you'll see:** "Native screen 1920x1080 -> scale 0.745 -> send 1430x804", then the model's (512, 300) click shown as WRONG on native (off by ~1.34x) versus RIGHT scaled back to (688, 403).

## 4 - The Anthropic Computer Use agent loop

Anthropic's Computer Use tool packages the vision loop as a schema-less API tool: you declare only the type, name, and display size, and Claude returns actions your sandbox executes. You own the loop - send prompt, get a `tool_use` action, run it, send a screenshot back as a `tool_result`, repeat until Claude stops or a cap fires. This cell scripts Claude so the loop runs with no key.

In [ ]:
# THE ANTHROPIC AGENT LOOP - schema-less computer-use tool, run in a loop.
# Claude returns tool_use blocks (action + coordinate); YOUR app executes them
# in a sandbox and sends a tool_result (a new screenshot) back. Repeat until
# Claude stops requesting tools OR a max-iteration cap fires. We script Claude.

def fake_claude(turn):
    # Stands in for client.beta.messages.create(...). Real Claude decides from a
    # screenshot; here a scripted plan returns the same tool_use shape.
    plan = [
        {"type": "tool_use", "action": "left_click", "coordinate": [640, 96]},   # open Invoices
        {"type": "tool_use", "action": "screenshot"},                            # look again
        {"type": "tool_use", "action": "type", "text": "invoice total"},         # search
        {"type": "text", "text": "The three invoices total 14,999."},            # done, no tool_use
    ]
    return plan[turn] if turn < len(plan) else {"type": "text", "text": "done"}

def execute(block):                            # your sandbox executes the requested action
    a = block["action"]
    if a == "screenshot":  return "screenshot(bytes)"
    if a == "left_click":  return f"clicked {block['coordinate']}"
    if a == "type":        return f"typed {block['text']!r}"
    return f"ran {a}"

def agent_loop(max_iterations=10):
    for i in range(max_iterations):
        block = fake_claude(i)
        if block["type"] != "tool_use":        # no tool requested -> task complete
            print(f"  iter {i}: Claude says -> {block['text']}")
            return {"iterations": i + 1, "done": True}
        result = execute(block)                # run it, send the result (screenshot) back
        print(f"  iter {i}: tool_use {block['action']:11} -> {result}")
    return {"iterations": max_iterations, "done": False}   # cap fired

print("Agent loop (tool_use -> execute -> screenshot back -> repeat):")
out = agent_loop()
print(f"\nStopped after {out['iterations']} iterations, done={out['done']}")
print("The max_iterations cap is the safeguard against an infinite, costly click loop.")
# Output:
# Agent loop (tool_use -> execute -> screenshot back -> repeat):
#   iter 0: tool_use left_click  -> clicked [640, 96]
#   iter 1: tool_use screenshot  -> screenshot(bytes)
#   iter 2: tool_use type        -> typed 'invoice total'
#   iter 3: Claude says -> The three invoices total 14,999.
#
# Stopped after 4 iterations, done=True
# The max_iterations cap is the safeguard against an infinite, costly click loop.

**Explanation**

A driver for the client-side agent loop, with a scripted stand-in for the real API call. `fake_claude` returns the same `tool_use` block shape the SDK would, `execute` plays your sandbox, and `agent_loop` runs the exchange until a text-only reply signals done. It teaches the wire pattern - who returns actions, who executes them - not any real screen control.

**How the code works - step by step**
- **`fake_claude()`** - stands in for `client.beta.messages.create(...)`; returns a scripted plan of `tool_use` blocks, then a final `text` block with no tool.
- **`execute()`** - your sandbox; runs the requested action (click / screenshot / type) and returns a result you would send back.
- **`agent_loop()`** - the loop: get a block, and if it is not a `tool_use`, the task is done; otherwise execute it and continue.
- **`max_iterations=10`** - the hard cap that guards against an infinite, costly click loop.

**In one line:** tool_use -> execute -> screenshot back -> repeat, until no tool or the cap fires.

**What you'll see:** Four iterations printing each action (left_click -> screenshot -> type 'invoice total' -> a final text answer "The three invoices total 14,999."), then "Stopped after 4 iterations, done=True" and a note that max_iterations is the safeguard.

## 5 - Perception cost: screenshot vs structure vs hybrid

How the agent *sees* is the biggest lever on cost and accuracy. A screenshot is universal but pricey (~ w*h/750 tokens); a distilled element list is far cheaper but fails on canvas and games. This cell prices the same task all three ways to show why 2026 systems go hybrid - and why a real loop needs prompt caching on top.

In [ ]:
# PERCEPTION COST - why hybrid perception beats screenshots-everywhere.
# A screenshot is universal but expensive; Anthropic's docs put it near (w*h)/750
# input tokens. A distilled element list is tiny - but only works on structured UIs.

def screenshot_tokens(w, h):
    return round(w * h / 750)          # Anthropic's documented approximation

def distilled_tokens(n_elements, per_element=30):
    return n_elements * per_element    # a numbered element list is a few tokens each

W, H = 1024, 768                       # recommended XGA screenshot
shot = screenshot_tokens(W, H)
dist = distilled_tokens(n_elements=4)
print(f"One {W}x{H} screenshot : ~{shot} tokens")
print(f"Distilled 4-element list: ~{dist} tokens  ({round(shot / dist)}x cheaper)")

steps = 20                              # a real task is many loop iterations
print(f"\nOver a {steps}-step task:")
print(f"  screenshot-every-step : ~{shot * steps:,} tokens")
print(f"  distilled-every-step  : ~{dist * steps:,} tokens")
# Hybrid: structure where it exists, a screenshot only when it doesn't (canvas/games).
vision_steps = 4                        # e.g. 4 steps need real pixels
hybrid = dist * (steps - vision_steps) + shot * vision_steps
print(f"  hybrid (16 distilled + 4 vision): ~{hybrid:,} tokens")
print("A11y/DOM alone fails on canvas, games, and poorly-accessible apps - so hybrid wins.")
# Output:
# One 1024x768 screenshot : ~1049 tokens
# Distilled 4-element list: ~120 tokens  (9x cheaper)
#
# Over a 20-step task:
#   screenshot-every-step : ~20,980 tokens
#   distilled-every-step  : ~2,400 tokens
#   hybrid (16 distilled + 4 vision): ~6,116 tokens
# A11y/DOM alone fails on canvas, games, and poorly-accessible apps - so hybrid wins.

**Explanation**

A pricing harness, not a model call. It applies Anthropic's documented screenshot-token approximation, compares it to a distilled element list, then scales both over a 20-step task and computes a hybrid mix. The takeaway is a number: structure is roughly 9x cheaper per step here, so default to it and escalate to pixels only when structure fails.

**How the code works - step by step**
- **`screenshot_tokens()`** - Anthropic's documented `(w*h)/750` approximation for one screenshot.
- **`distilled_tokens()`** - a numbered element list costed at a few tokens each.
- **Single-step compare** - one 1024x768 screenshot vs a 4-element list, showing the ~9x gap.
- **20-step scaling** - screenshot-every-step vs distilled-every-step over a realistic task.
- **`hybrid`** - structure for most steps plus a screenshot only for the few that need real pixels.

**In one line:** default to the cheapest modality that works, escalate to a screenshot only when structure fails.

**What you'll see:** "One 1024x768 screenshot : ~1049 tokens" vs "Distilled 4-element list: ~120 tokens (9x cheaper)", then a 20-step comparison (~20,980 vs ~2,400 vs ~6,116 hybrid) and a note that a11y/DOM alone fails on canvas and games.

## 6 - The safety gate

An agent with your mouse and keyboard needs defense in depth: sandbox, domain allowlist, human-in-the-loop on irreversible actions, a step-cap kill switch, and prompt-injection defense. Safety is the first design decision, not the last. This cell runs every proposed action through a gate before it can execute.

In [ ]:
# THE SAFETY GATE - never let a proposed action run unchecked.
# Defense in depth: domain allowlist -> destructive-action check (HITL) ->
# step cap -> prompt-injection flag. Each proposed action passes the gate first.

ALLOWLIST = {"portal", "wikipedia.org"}
DESTRUCTIVE = {"delete", "purchase", "send_email", "submit_payment"}

def domain_of(action):
    return action.get("url", "portal").split("/")[0]

def gate(action, step, max_steps=25, injection_flagged=False):
    if step > max_steps:
        return ("KILL", "step cap exceeded - stop the agent")
    if injection_flagged:
        return ("ESCALATE", "possible prompt injection in screenshot - confirm with human")
    if action.get("url") and domain_of(action) not in ALLOWLIST:
        return ("BLOCK", f"domain '{domain_of(action)}' not in allowlist")
    if action["type"] in DESTRUCTIVE:
        return ("ESCALATE", f"'{action['type']}' is irreversible - require human approval")
    return ("ALLOW", "safe, reversible, on-allowlist")

proposed = [
    {"type": "click",    "url": "portal/invoices"},
    {"type": "goto",     "url": "evil.com/steal"},              # off allowlist
    {"type": "purchase", "url": "portal/checkout"},             # irreversible
    {"type": "type",     "url": "portal/search"},
    {"type": "click",    "url": "portal/ok", "_inj": True},     # injection seen in screenshot
]
print("Every proposed action passes the safety gate first:")
for i, a in enumerate(proposed, 1):
    verdict, why = gate(a, step=i, injection_flagged=a.get("_inj", False))
    print(f"  {a['type']:9} {domain_of(a):9} -> {verdict:8} ({why})")
print("\nOnly ALLOW runs automatically; ESCALATE waits for a human; BLOCK/KILL never run.")
# Output:
# Every proposed action passes the safety gate first:
#   click     portal    -> ALLOW    (safe, reversible, on-allowlist)
#   goto      evil.com  -> BLOCK    (domain 'evil.com' not in allowlist)
#   purchase  portal    -> ESCALATE ('purchase' is irreversible - require human approval)
#   type      portal    -> ALLOW    (safe, reversible, on-allowlist)
#   click     portal    -> ESCALATE (possible prompt injection in screenshot - confirm with human)
#
# Only ALLOW runs automatically; ESCALATE waits for a human; BLOCK/KILL never run.

**Explanation**

An action-authorization gate, not a model call. Each proposed action is checked in priority order - kill on cap, escalate on injection, block off-allowlist domains, escalate irreversible actions - and only clearly safe, reversible, on-allowlist actions get an automatic ALLOW. It encodes defense-in-depth as a single guard every action must pass.

**How the code works - step by step**
- **`ALLOWLIST` / `DESTRUCTIVE`** - the approved domains and the set of irreversible action types.
- **`domain_of()`** - extracts the domain from an action's URL to check against the allowlist.
- **`gate()`** - the guard, checked top-down: step cap -> KILL, injection flag -> ESCALATE, off-allowlist -> BLOCK, destructive -> ESCALATE, else ALLOW.
- **`proposed`** - a batch of actions covering each outcome (safe click, off-allowlist goto, purchase, safe type, injection-flagged click).
- **Loop** - runs every action through the gate and prints the verdict and reason.

**In one line:** every action passes the gate first; only ALLOW runs automatically, ESCALATE waits for a human, BLOCK/KILL never run.

**What you'll see:** Five verdicts - portal click ALLOW, evil.com goto BLOCK, purchase ESCALATE, portal type ALLOW, injection-flagged click ESCALATE - then "Only ALLOW runs automatically; ESCALATE waits for a human; BLOCK/KILL never run."

## 7 - The API-first hierarchy

The judgment that ties the lesson together: the approaches form a hierarchy - official API > DOM automation > vision + coordinates > native computer use - and you always pick the highest one that works. Higher is cheaper and more reliable; lower is more general. This cell routes a set of tasks down that hierarchy with a reason and a relative cost tier.

In [ ]:
# THE API-FIRST HIERARCHY - choose the cheapest reliable approach that works.
# Prefer, in order: official API > DOM/browser automation > vision+coords > native CUA.
# Higher = cheaper and more reliable; lower = more general. Climb down only when forced.

def choose(has_api, is_browser, structured_dom, needs_desktop):
    if has_api:
        return ("Official API", "$", "there is a real API - use the front door")
    if is_browser and structured_dom:
        return ("DOM automation", "$$", "browser with a parseable DOM - act by element")
    if is_browser and not structured_dom:
        return ("Vision + coordinates", "$$$", "canvas/custom UI the DOM cannot describe")
    if needs_desktop:
        return ("Native computer use", "$$$$", "a desktop app beyond the browser")
    return ("Vision + coordinates", "$$$", "no API and no clean DOM - fall back to pixels")

cases = [
    ("Fetch orders from a billing system", dict(has_api=True,  is_browser=True,  structured_dom=True,  needs_desktop=False)),
    ("Fill a form on a normal web app",    dict(has_api=False, is_browser=True,  structured_dom=True,  needs_desktop=False)),
    ("Play a canvas-based web game",       dict(has_api=False, is_browser=True,  structured_dom=False, needs_desktop=False)),
    ("Drive a legacy desktop app",         dict(has_api=False, is_browser=False, structured_dom=False, needs_desktop=True)),
]
print("Route each task to the cheapest approach that works:")
for task, props in cases:
    approach, cost, why = choose(**props)
    print(f"  {task:36} -> {cost:5} {approach:20} ({why})")
print("\nReach for computer use LAST - it is the most general and the most expensive.")
# Output:
# Route each task to the cheapest approach that works:
#   Fetch orders from a billing system   -> $     Official API         (there is a real API - use the front door)
#   Fill a form on a normal web app      -> $$    DOM automation       (browser with a parseable DOM - act by element)
#   Play a canvas-based web game         -> $$$   Vision + coordinates (canvas/custom UI the DOM cannot describe)
#   Drive a legacy desktop app           -> $$$$  Native computer use  (a desktop app beyond the browser)
#
# Reach for computer use LAST - it is the most general and the most expensive.

**Explanation**

A decision router, not a model call. `choose` takes four booleans about a task and returns the highest-priority approach that fits, its cost tier, and why. It makes the whole lesson's advice mechanical: use the front door (API) if there is one, and climb down to pixels only when forced.

**How the code works - step by step**
- **`choose()`** - checks top-down: has_api -> Official API ($), browser with structured DOM -> DOM automation ($$), browser without -> Vision + coordinates ($$$), else desktop -> Native computer use ($$$$).
- **`cases`** - four tasks spanning the full range (billing API, normal web form, canvas game, legacy desktop app).
- **Loop** - routes each task and prints its cost tier, chosen approach, and reason.

**In one line:** pick the highest-priced-reliability approach that works; reach for native computer use last.

**What you'll see:** Four routed tasks - billing API -> $ Official API, web form -> $$ DOM automation, canvas game -> $$$ Vision + coordinates, legacy desktop -> $$$$ Native computer use - then "Reach for computer use LAST - it is the most general and the most expensive."

You now have every moving part of a computer-use agent in code: the observe-think-act loop (Cell 1), the two ways to see a browser page cheaply - DOM distillation (Cell 2) - and to see anything at all - vision with coordinate scaling (Cell 3), the native Anthropic agent loop that packages it (Cell 4), the perception-cost lever that makes hybrid win (Cell 5), the safety gate every action must pass (Cell 6), and the API-first hierarchy that tells you which to pick (Cell 7). The through-line: computer use is the most general capability and the most expensive and fragile, so climb down to it last, sandbox it always, and gate the irreversible actions. Next up is evaluating and securing these agents (Lesson 11.11) and human-in-the-loop approval flows (Lesson 11.10).